# `dlmrel` on Colab A100 — GPU stages

Runs the two GPU stages of the pipeline: **`search`** (score every head on every
relation) and **`curve`** (masked-state accuracy over diffusion time).

`data`, `nulls` and `analyze` are CPU stages and run locally — see HANDOFF.md §7.

## Before you start

Run **`dlmrel data`** and **`dlmrel nulls`** locally first and read the output.
They are free and they gate everything: if the nulls under random full-treebank
sampling come out far from HANDOFF §4.1, every downstream claim moves and this
GPU run may not be worth booking.

Then zip the project locally (the helper at the bottom of this notebook prints
the exact command) so the upload carries **both** the package *and* the
`results/<run>/sentences_*.csv` manifests produced by `dlmrel data`.
The manifests must travel: `splits.py` rebuilds splits from config and asserts
against them, which is the guard against config drift between stages
(HANDOFF §9, gotcha 8). Regenerating them here in the same session would make
that assert check its own output and prove nothing.

## Runtime, read this

Neither stage batches — one sentence per forward pass. Memory is not the
constraint on an 80GB A100; wall-clock is. `curve` at full config is
`5 seeds x 64 timesteps x 1000 sentences = 320,000` forward passes and will very
likely outlast a Colab session. Cell 8 measures your actual throughput and
extrapolates **before** you commit to the long run. Do not skip it.

## 1 · GPU check

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > A100.'
props = torch.cuda.get_device_properties(0)
print(f'\n{props.name}  |  {props.total_memory / 1e9:.0f} GB')
if 'A100' not in props.name:
    print('WARNING: expected an A100. Timings below assume one.')

## 2 · Upload the project zip

Pick the `dlmrel-upload.zip` built by the last cell of this notebook (or by the
equivalent command run locally).

In [ ]:
import shutil, pathlib
from google.colab import files

WORK = pathlib.Path('/content/dlmresearch')
if WORK.exists():
    shutil.rmtree(WORK)

uploaded = files.upload()
assert uploaded, 'nothing uploaded'
zip_name = next(iter(uploaded))
print(f'\nunpacking {zip_name} ({len(uploaded[zip_name]) / 1e6:.2f} MB)')

WORK.mkdir(parents=True)
shutil.unpack_archive(zip_name, WORK)

# Tolerate a zip that wraps everything in one top-level folder.
entries = [p for p in WORK.iterdir() if not p.name.startswith('.')]
if len(entries) == 1 and entries[0].is_dir() and not (WORK / 'pyproject.toml').exists():
    inner = entries[0]
    for p in inner.iterdir():
        shutil.move(str(p), str(WORK / p.name))
    inner.rmdir()

%cd /content/dlmresearch
!ls -a

## 3 · Install

`transformers` is pinned to **4.44.2** by `pyproject.toml [gpu]` — later versions
changed the attention-implementation plumbing that `output_attentions` depends
on (HANDOFF §7). Colab ships a much newer one, so this downgrade is mandatory,
not optional.

**Colab will tell you to restart the runtime after this cell. Do not.** Restarting
wipes `/content` state you just set up; the import check in cell 4 is what
actually decides whether the install took.

In [ ]:
!pip install -q -e ".[gpu]" 2>&1 | tail -20

## 4 · Verify the install

If `transformers` is not 4.44.2, stop — every accuracy downstream can silently
become zero.

In [ ]:
import importlib, transformers, dlmrel
importlib.reload(transformers)

print('transformers', transformers.__version__)
assert transformers.__version__ == '4.44.2', (
    f'expected 4.44.2, got {transformers.__version__} — rerun cell 3'
)

from dlmrel.config import Config
CONFIG_PATH = 'configs/default.yaml'
cfg = Config.load(CONFIG_PATH)
print('model      ', cfg.model.name)
print('attn impl  ', cfg.model.attn_implementation)
print('out_dir    ', cfg.out_dir)
print('steps      ', cfg.diffusion.steps)
print('seeds      ', cfg.diffusion.seeds)
assert cfg.model.attn_implementation == 'eager', (
    'attn_implementation must be eager — sdpa and flash_attention_2 accept '
    'output_attentions=True and return nothing (HANDOFF §9, gotcha 1)'
)

## 5 · Confirm the split manifests came along

These are what `splits.py` asserts against. Without them the GPU stages would
score a different set of sentences than `dlmrel data` recorded, silently.

In [ ]:
import pandas as pd, pathlib

OUT = pathlib.Path(cfg.out_dir)
missing = [
    n for n in ('sentences_select.csv', 'sentences_dev.csv', 'sentences_test.csv')
    if not (OUT / n).exists()
]
assert not missing, (
    f'missing manifests in {OUT}: {missing}\n'
    'Run `dlmrel data` locally and re-zip, including results/. '
    'Do NOT regenerate them here — the drift guard would then be checking '
    'its own output.'
)
for n in ('select', 'dev', 'test'):
    print(f'{n:7s} {len(pd.read_csv(OUT / f"sentences_{n}.csv")):5d} sentences')

inst_path = OUT / 'relation_instances.csv'
if inst_path.exists():
    inst = pd.read_csv(inst_path)
    print(f'\n{len(inst)} relation instances')
    print(pd.crosstab(inst['relation'], inst['split']))

## 6 · Load the model

Clones `HKUNLP/DiffuLLaMA` into `third_party/` (not on PyPI) and runs a load-time
assertion that attentions actually come back. Expect a few minutes for the 7B
weights on first run.

In [ ]:
import time
from dlmrel.model import load_model

t0 = time.time()
model, tokenizer, meta = load_model(cfg.model)
print(f'\nloaded in {time.time() - t0:.0f}s')
print(meta)
print(f'GPU allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

## 7 · Rebuild the splits

Examples hold tokenizer-dependent spans, so they are rebuilt rather than
serialised. This asserts against the manifests from cell 5 — if the config
drifted since `dlmrel data`, it raises here rather than scoring the wrong
sentences.

In [ ]:
from dlmrel.splits import build_all_splits

# `examples_for_split` reloads and re-tokenizes the whole treebank on every
# call, so building all three at once avoids doing that work three times.
# The manifest assert below is the same guard it applies internally.
t0 = time.time()
splits = build_all_splits(cfg, tokenizer)
print(f'built in {time.time() - t0:.0f}s\n')

for name in ('select', 'dev', 'test'):
    expected = pd.read_csv(OUT / f'sentences_{name}.csv')['sentence'].tolist()
    actual = [e.text for e in splits[name]]
    if expected != actual:
        raise RuntimeError(
            f'split {name!r} does not match sentences_{name}.csv: '
            f'{len(expected)} recorded vs {len(actual)} rebuilt. '
            'The config changed since `dlmrel data` ran — rerun it locally '
            'and re-zip.'
        )
    n_inst = sum(len(e.relations) for e in splits[name])
    print(f'{name:7s} {len(splits[name]):5d} sentences  {n_inst:6d} instances  ✓ matches manifest')

## 8 · Measure throughput before committing

Times a small sample and extrapolates. **Read the estimate before running
cells 9 and 10.**

In [ ]:
from dlmrel.scoring import score_split

PROBE_N = 25
probe = splits['test'][:PROBE_N]

torch.cuda.synchronize(); t0 = time.time()
_ = score_split(model, tokenizer, probe, cfg.diffusion, 'probe', log_every=0)
torch.cuda.synchronize()
per_sentence = (time.time() - t0) / PROBE_N

n_search = sum(len(v) for v in splits.values())
search_h = per_sentence * n_search / 3600

print(f'\nper sentence      {per_sentence * 1000:.0f} ms')
print(f'search  {n_search} sentences -> {search_h:.2f} h')

n_test = len(splits['test'])
print(f'\ncurve cost = seeds x timesteps x {n_test} sentences')
print(f'{"seeds":>6} {"stride":>7} {"timesteps":>10} {"forwards":>10} {"est. hours":>11}')
for n_seeds in (1, 3, 5):
    for stride in (1, 4, 8):
        n_t = len(range(0, cfg.diffusion.steps, stride))
        fwd = n_seeds * n_t * n_test
        print(f'{n_seeds:>6} {stride:>7} {n_t:>10} {fwd:>10,} {per_sentence * fwd / 3600:>10.1f}h')

## 9 · `search`

Scores every (layer, head) on every relation at the final, fully-revealed frame
and writes `head_scores_{select,dev,test}.csv` plus the merged join everything
downstream reads.

In [ ]:
import json
from dlmrel.scoring import merge_splits

OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'model_meta.json').write_text(json.dumps(meta, indent=2))

frames = {}
for name in ('select', 'dev', 'test'):
    t0 = time.time()
    frames[name] = score_split(model, tokenizer, splits[name], cfg.diffusion, name)
    frames[name].to_csv(OUT / f'head_scores_{name}.csv', index=False)
    print(f'[{name}] done in {(time.time() - t0) / 60:.1f} min')

merged = merge_splits(frames['select'], frames['test'], frames['dev'])
merged.to_csv(OUT / 'head_scores_merged.csv', index=False)
print(f'\nmerged -> {OUT / "head_scores_merged.csv"}  ({len(merged)} rows)')

# Cheap sanity check: all-zero accuracy is the signature of attentions not
# being returned (HANDOFF §9, gotcha 1).
assert merged['accuracy_select'].max() > 0, (
    'every head scored zero — attentions were not returned'
)
print('\nbest head per relation on the selection split:')
for rel, grp in merged.groupby('relation'):
    b = grp.sort_values('accuracy_select', ascending=False).iloc[0]
    print(f"  {rel:22s} L{int(b['layer']):02d} H{int(b['head']):02d} "
          f"select={b['accuracy_select']:.3f} test={b['accuracy_test']:.3f}")

## 10 · `curve` (optional, expensive)

Set the scope from the cell 8 table. Notes on the two knobs:

- **`N_SEEDS`** — HANDOFF §4.3 says the 0.434-vs-0.108 comparison is *not*
  protocol-matched precisely because DiffuGPT-S used 5 seeds and the 7B run used
  1. Dropping to `N_SEEDS=1` recreates that exact defect. Keep 5 if this run is
  meant to settle open question 2.
- **`TIMESTEP_STRIDE`** — cheaper and safer. The masked statistic is gated to
  frames with `>= min_masked_positions` (25) still masked, which are the *early*
  timesteps; late ones contribute only to the unmasked statistic. Subsampling
  timesteps thins the curve without biasing either statistic.

Writes `curve_raw.csv` one row per (relation, seed, timestep, instance) —
deliberately not pre-aggregated, so the null can later be recomputed over
exactly the instances that entered the average.

In [ ]:
N_SEEDS = 5           # from cfg.diffusion.seeds
TIMESTEP_STRIDE = 4   # 1 = every timestep

from dlmrel.scoring import masked_state_curve, aggregate_curve

merged = pd.read_csv(OUT / 'head_scores_merged.csv')
heads = {
    rel: (int(b['layer']), int(b['head']))
    for rel, grp in merged.groupby('relation')
    for b in [grp.sort_values('accuracy_select', ascending=False).iloc[0]]
}
seeds = cfg.diffusion.seeds[:N_SEEDS]
timesteps = list(range(0, cfg.diffusion.steps, TIMESTEP_STRIDE))
print(f'heads     {heads}')
print(f'seeds     {seeds}')
print(f'timesteps {len(timesteps)} of {cfg.diffusion.steps}')

t0 = time.time()
raw = masked_state_curve(
    model, tokenizer, splits['test'], heads, cfg.diffusion,
    seeds=seeds, timesteps=timesteps,
)
raw.to_csv(OUT / 'curve_raw.csv', index=False)
print(f'\ncurve done in {(time.time() - t0) / 60:.1f} min  ({len(raw)} rows)')

agg = aggregate_curve(raw, cfg.diffusion.min_masked_positions)
agg.to_csv(OUT / 'curve_aggregate.csv', index=False)
print(agg.to_string(index=False))

## 11 · Zip results and download

Only `results/` comes back — the code went up from your machine and is unchanged.
Unpack over the local `results/` directory, then run `dlmrel analyze` locally.

In [ ]:
import datetime

stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M')
archive = f'/content/dlmrel-results-{stamp}'
shutil.make_archive(archive, 'zip', root_dir='.', base_dir='results')

size = pathlib.Path(archive + '.zip').stat().st_size
print(f'{archive}.zip  ({size / 1e6:.1f} MB)')
for p in sorted(OUT.iterdir()):
    print(f'  {p.name:32s} {p.stat().st_size / 1e6:8.2f} MB')

files.download(archive + '.zip')

---

## Appendix · building the upload zip locally

Run this on your machine, not here:

```bash
cd ~/dlmresearch
zip -r ~/Downloads/dlmrel-upload.zip . \
  -x '.venv/*' '.git/*' '*/__pycache__/*' '.pytest_cache/*' '.ruff_cache/*' \
     '*.egg-info/*' '*.DS_Store' 'data/*' 'third_party/*' 'notebooks/*'
```

That comes to roughly **55 KB** plus whatever `results/` holds — the upload is
trivial, so rebuild it fresh each session rather than reusing a stale zip.

`data/` and `third_party/` are excluded because they are re-downloaded on Colab.
`results/` is **kept** — that is where the split manifests live.

## After the download

```bash
cd ~/dlmresearch
unzip -o ~/Downloads/dlmrel-results-*.zip
.venv/bin/dlmrel analyze --config configs/default.yaml
```